# 🎵 IPT Recognition — Google Colab

Train your own **Instrumental Playing Techniques** classification model directly in the browser, no local install required.

This notebook mirrors `train.ipynb` but runs everything on Google Colab.

**Before you start**
1. Set the runtime to GPU: `Runtime` ▸ `Change runtime type` ▸ **T4 GPU** (or any GPU).
2. Organize your audio files so that each technique has its own sub-folder:

```
my_data_folder/
├── IPTclass_1/
│   ├── audiofile1.wav
│   └── audiofile2.wav
└── IPTclass_2/
    ├── audiofile1.wav
    └── audiofile2.wav
```

3. Upload `my_data_folder` to your Google Drive (you'll connect it below).

Then run the cells top to bottom. ▶️

## 1. Check GPU

In [ ]:
#@title Verify the GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run.")


## 2. Get the code

In [ ]:
#@title Clone the repository and install requirements
import os

REPO_URL = "https://github.com/nbrochec/ipt_recognition.git"
BRANCH   = "performance"  #@param {type:"string"}

if not os.path.isdir("ipt_recognition"):
    !git clone --branch {BRANCH} {REPO_URL}
%cd ipt_recognition
!pip install -q -r requirements.txt
print("\nDone. Working directory:", os.getcwd())


## 3. Connect your data

Mount your Google Drive, then point to the folder that holds your class sub-folders.
The folder is copied into `data/raw/` where the pipeline expects it.

In [ ]:
#@title Mount Google Drive and import your audio folder
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

#@markdown Path to your data folder inside Google Drive (the folder that contains one sub-folder per IPT class):
drive_data_path = "/content/drive/MyDrive/my_data_folder"  #@param {type:"string"}

#@markdown Name to give this folder inside the project (used as `--train_dir`):
train_soundfiles = "my_data_folder"  #@param {type:"string"}

dest = os.path.join("data", "raw", train_soundfiles)
assert os.path.isdir(drive_data_path), f"Not found: {drive_data_path}"
os.makedirs(os.path.dirname(dest), exist_ok=True)
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree(drive_data_path, dest)

classes = sorted(d for d in os.listdir(dest) if os.path.isdir(os.path.join(dest, d)))
print(f"Copied to {dest}")
print(f"Found {len(classes)} class folder(s):", classes)


## 4. Preprocess your dataset

In [ ]:
#@title Preprocessing parameters
run_name = "my_run"  #@param {type:"string"}
#@markdown Sampling rate in Hz (downsampling is recommended):
sr = 24000  #@param {type:"integer"}
#@markdown Classification window length. Use `ms` for milliseconds or `samps` for samples.
#@markdown Larger windows suit longer techniques (e.g. legato).
segment_length = "1000 ms"  #@param {type:"string"}


In [ ]:
#@title Run preprocessing
!python preprocess.py --name {run_name} --sampling_rate {sr} --train_dir {train_soundfiles} -seglen "{segment_length}"


## 5. Train your model

In [ ]:
#@title Training parameters
#@markdown `cuda` uses the Colab GPU (recommended). Use `cpu` only if no GPU is available.
device = "cuda"  #@param ["cuda", "cpu"]
#@markdown Model architecture (both work for other instruments too):
model = "flute"  #@param ["flute", "eguitar"]
epochs = 100  #@param {type:"integer"}
#@markdown Stop training after this many epochs without improvement:
early_stopping = 10  #@param {type:"integer"}
#@markdown Online data augmentation (polarity inversion, low/high-pass filters):
online_augment = 1  #@param [0, 1] {type:"raw"}


In [ ]:
#@title Run training
!python train.py --name {run_name} --sampling_rate {sr} -seglen "{segment_length}" --device {device} --model {model} --epochs {epochs} --online_augment {online_augment} --early_stopping {early_stopping}


## 6. Get your trained model

Your run is saved under `runs/<run_name>_<date_time>/` and contains the checkpoint
(`.pth`), the TorchScript model (`.ts`) for use in Max/MSP with `ipt~`, and the config.

The cell below finds the latest run and copies it to your Google Drive so it survives
after the Colab session ends.

In [ ]:
#@title Save the latest run to Google Drive
import glob, os, shutil

runs = sorted(glob.glob("runs/*"), key=os.path.getmtime)
assert runs, "No runs found. Did training finish?"
latest = runs[-1]
print("Latest run:", latest)
for f in sorted(os.listdir(latest)):
    print("  -", f)

#@markdown Destination folder in your Google Drive:
drive_output_dir = "/content/drive/MyDrive/ipt_runs"  #@param {type:"string"}
os.makedirs(drive_output_dir, exist_ok=True)
dest = os.path.join(drive_output_dir, os.path.basename(latest))
if os.path.isdir(dest):
    shutil.rmtree(dest)
shutil.copytree(latest, dest)
print("\nSaved to:", dest)
